# 06. Visual Embeddings with SigLIP (neutral dataset)

This notebook generates SigLIP embeddings for the neutral-imputed dataset. Unlike the strict dataset, this version retains a broader set of artworks and replaces missing warmth and competence values with a neutral score of 0.5.

The goal is to evaluate whether predictive performance changes when the same visual encoder is trained against a larger but noisier target set.

In [6]:
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from PIL import Image
from tqdm import tqdm
from transformers import AutoProcessor, AutoModel

/home/agrupa-lab/agrupa/IE_capstones/Naji/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
# ----------------------------
# Configuration
# ----------------------------
print("🔧 Loading configuration...")

DATASET_CSV = "../Data/Processed/final_dataset_neutral.csv"
MODEL_NAME = "google/siglip-base-patch16-224"

OUTPUT_DIR = Path("../Embeddings/siglip")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_EMBEDDINGS = OUTPUT_DIR / "siglip_neutral_embeddings.npy"
OUTPUT_IDS = OUTPUT_DIR / "siglip_neutral_catnos.csv"
OUTPUT_MERGED = OUTPUT_DIR / "final_dataset_neutral_with_siglip.csv"
OUTPUT_FAILED = OUTPUT_DIR / "siglip_neutral_failed_paths.csv"

BATCH_SIZE = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("📁 Dataset:", DATASET_CSV)
print("🧠 Model:", MODEL_NAME)
print("💾 Output dir:", OUTPUT_DIR)
print("⚙️ Batch size:", BATCH_SIZE)
print("🚀 Using device:", device)
print("✅ Configuration loaded\n")

🔧 Loading configuration...
📁 Dataset: ../Data/Processed/final_dataset_neutral.csv
🧠 Model: google/siglip-base-patch16-224
💾 Output dir: ../Embeddings/siglip
⚙️ Batch size: 16
🚀 Using device: cuda
✅ Configuration loaded



## 1. Loading the neutral dataset

The neutral-imputed dataset is used as the input for embedding extraction. Before generating embeddings, image paths are resolved to their full location on disk so that only valid files are processed.

In [21]:
print("📥 Loading dataset...")

df = pd.read_csv(DATASET_CSV)

print("✅ Dataset loaded")
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head(2)

📥 Loading dataset...
✅ Dataset loaded
Dataset shape: (6856, 21)

Columns:
['cat_no', 'titulo', 'autor', 'escuela_obra', 'tipo_objeto', 'datacion', 'tema', 'is_religious', 'is_fauna', 'century', 'image_path', 'descripcion', 'animal_cluster', 'n_descriptores_fila', 'n_en_diccionario_fila', 'dirmean_Warmth', 'n_dirmean_Warmth', 'dirmean_Competence', 'n_dirmean_Competence', 'warmth_was_missing', 'competence_was_missing']


,cat_no,titulo,autor,escuela_obra,tipo_objeto,datacion,tema,is_religious,is_fauna,century,...,descripcion,animal_cluster,n_descriptores_fila,n_en_diccionario_fila,dirmean_Warmth,n_dirmean_Warmth,dirmean_Competence,n_dirmean_Competence,warmth_was_missing,competence_was_missing
0,P000002,El juicio de Paris,"Albani, Francesco",Italiana,Pintura,1650 - 1660,NaN,0,1,17th c.,...,"La obra de Francesco Albani, uno de los discíp...",purity,890.0,62.0,0.333333,9,0.7,10,0,0
1,P000006,Sagrada Familia y el cardenal Fernando de Medici,"Allori, Alessandro",Italiana,Pintura,1584,NaN,1,1,16th c.,...,"San José, la Virgen con el Niño en brazos y Sa...",other,148.0,17.0,0.937500,4,0.5,2,0,0


In [22]:
# --------------------------------------------------
# Fix image paths + check availability
# --------------------------------------------------

print("📂 Fixing image paths...")

RAW_IMAGE_ROOT = Path("/home/agrupa-lab/agrupa/data_raw")

df["full_path"] = df["image_path"].apply(
    lambda p: str(RAW_IMAGE_ROOT / p) if isinstance(p, str) else None
)

df["file_exists"] = df["full_path"].apply(
    lambda p: Path(p).exists() if isinstance(p, str) else False
)

total_rows = len(df)
missing_images = int((~df["file_exists"]).sum())

print("\n📊 Image availability check:")
print("Total rows:", total_rows)
print("Missing images:", missing_images)

df = df[df["file_exists"]].copy().reset_index(drop=True)

print("Rows used for embeddings:", len(df))
print("\n✅ Path fixing complete\n")

📂 Fixing image paths...

📊 Image availability check:
Total rows: 6856
Missing images: 751
Rows used for embeddings: 6105

✅ Path fixing complete



## 2. Loading the SigLIP model

SigLIP is loaded in evaluation mode and used only as a frozen feature extractor. No fine-tuning is applied at this stage; the objective is to obtain stable pre-trained image representations for later regression experiments.

In [23]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.to(device)
_ = model.eval()

print("✅ SigLIP model loaded")

Loading weights: 100%|██████████| 408/408 [00:00<00:00, 13740.77it/s]


✅ SigLIP model loaded


In [24]:
def load_image(path):
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return None

## 3. Extracting embeddings in batches

Embeddings are generated in batches to improve efficiency and reduce memory issues. For each valid image, the output vector is stored together with its corresponding `cat_no`, allowing the embeddings to be aligned back to the dataset used for modeling.

In [25]:
print("🖼️ Starting embedding extraction...")
print("Total batches:", len(df) // BATCH_SIZE + 1)

all_embeddings = []
all_catnos = []
failed_paths = []

for i in tqdm(range(0, len(df), BATCH_SIZE), desc="🔄 Extracting SigLIP embeddings"):
    batch = df.iloc[i:i + BATCH_SIZE]

    images = []
    catnos = []

    for _, row in batch.iterrows():
        img = load_image(row["full_path"])
        if img is None:
            failed_paths.append(row["full_path"])
            continue
        images.append(img)
        catnos.append(row["cat_no"])

    if len(images) == 0:
        continue

    inputs = processor(images=images, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.get_image_features(**inputs)

    if hasattr(out, "image_embeds"):
        emb = out.image_embeds
    elif hasattr(out, "pooler_output"):
        emb = out.pooler_output
    elif hasattr(out, "last_hidden_state"):
        emb = out.last_hidden_state[:, 0, :]
    else:
        emb = out

    emb = emb.detach().cpu().numpy()

    all_embeddings.append(emb)
    all_catnos.extend(catnos)

if len(all_embeddings) == 0:
    raise RuntimeError("No embeddings extracted.")

embeddings = np.vstack(all_embeddings)
ids_df = pd.DataFrame({"cat_no": all_catnos})

print("\nEmbedding shape:", embeddings.shape)
print("IDs count:", len(ids_df))
print("Failed image loads:", len(failed_paths))

🖼️ Starting embedding extraction...
Total batches: 382


🔄 Extracting SigLIP embeddings: 100%|██████████| 382/382 [01:09<00:00,  5.48it/s]


Embedding shape: (6105, 768)
IDs count: 6105
Failed image loads: 0


In [26]:
# Save raw embeddings
np.save(OUTPUT_EMBEDDINGS, embeddings)
ids_df.to_csv(OUTPUT_IDS, index=False)

# Save merged dataset with embedding columns
emb_cols = [f"emb_{i}" for i in range(embeddings.shape[1])]
emb_df = pd.DataFrame(embeddings, columns=emb_cols)

merged_emb = pd.concat(
    [ids_df.reset_index(drop=True), emb_df.reset_index(drop=True)],
    axis=1
)

final_df = df.merge(merged_emb, on="cat_no", how="inner")
final_df.to_csv(OUTPUT_MERGED, index=False)

if failed_paths:
    pd.DataFrame({"failed_path": failed_paths}).to_csv(OUTPUT_FAILED, index=False)

print("\nSaved files:")
print("-", OUTPUT_EMBEDDINGS)
print("-", OUTPUT_IDS)
print("-", OUTPUT_MERGED)
if failed_paths:
    print("-", OUTPUT_FAILED)


Saved files:
- ../Embeddings/siglip/siglip_neutral_embeddings.npy
- ../Embeddings/siglip/siglip_neutral_catnos.csv
- ../Embeddings/siglip/final_dataset_neutral_with_siglip.csv


## 4. Quick validation of saved outputs

Before moving to the modeling stage, the saved outputs are checked to confirm that the number of embeddings, IDs, and merged dataset rows are aligned correctly.

In [27]:
emb = np.load(OUTPUT_EMBEDDINGS)
ids = pd.read_csv(OUTPUT_IDS)
merged_df = pd.read_csv(OUTPUT_MERGED)

print("Embeddings shape:", emb.shape)
print("IDs shape:", ids.shape)
print("Merged dataset shape:", merged_df.shape)

print("\nUnique cat_no in IDs:", ids["cat_no"].nunique())
print("Unique cat_no in merged dataset:", merged_df["cat_no"].nunique())

print("\nEmbedding stats:")
print("Mean:", emb.mean())
print("Std:", emb.std())
print("Min:", emb.min())
print("Max:", emb.max())

Embeddings shape: (6105, 768)
IDs shape: (6105, 1)
Merged dataset shape: (6105, 791)

Unique cat_no in IDs: 6105
Unique cat_no in merged dataset: 6105

Embedding stats:
Mean: 0.036465026
Std: 0.522763
Min: -4.100309
Max: 7.2785964


## Interpretation of SigLIP embeddings (neutral dataset)

The embedding extraction process was successfully completed for 6,105 artworks from the neutral-imputed dataset. Compared to the strict dataset, this represents a substantially larger sample, reflecting the more inclusive filtering strategy used in earlier preprocessing steps.

The embeddings have a dimensionality of 768, consistent with the SigLIP architecture. All images were successfully processed without failures, indicating that the image path resolution and preprocessing pipeline are robust.

From a distributional perspective, the embedding values appear well-behaved, with a mean close to zero (0.036) and a moderate standard deviation (0.523). This suggests that the representations are centered and scaled in a way that is suitable for downstream machine learning models.

The larger dataset size is expected to provide broader visual coverage of the Prado collection, including artworks with weaker or less reliable textual signals. This makes the neutral dataset particularly useful for evaluating whether visual embeddings alone can capture stereotype-related information, even when the textual supervision is noisy or partially imputed.

However, this increased coverage comes at a cost: the target variables (warmth and competence) are less reliable due to the inclusion of imputed values. As a result, any improvement or degradation in model performance relative to the strict dataset will reflect the trade-off between data quantity and label quality.

Overall, this dataset provides a complementary perspective to the strict dataset, enabling a more comprehensive evaluation of the relationship between visual representations and social stereotype dimensions.